<a href="https://colab.research.google.com/github/smile-rr/ocr-fine-app/blob/main/notebooks/03b_stage2_colab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03b · Stage 2 · LLM QLoRA（Colab · GPU）

本 notebook 在 Colab 跑 **Qwen2.5-0.5B-Instruct** 的 4-bit QLoRA 表格问答微调，用 **Unsloth**。

**硬件**：T4 GPU 足够（0.5B 模型在 4-bit 下不到 1 GB）。

**流程**：装依赖 → HF 登录 → 上传 `stage2_colab.zip`（~1–5 MB，纯 jsonl）→ 加载模型 → 训练 → 下载 adapter。


## 1. 检查 GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('CUDA:', torch.cuda.is_available(), '| bf16:', torch.cuda.is_bf16_supported())

## 2. 安装依赖

In [ ]:
%%capture
!pip install -U unsloth
!pip install -U datasets trl peft accelerate bitsandbytes

## 3. HF 登录

两种方式（任选）：
- Colab Secret（🔑 侧栏加 `HF_TOKEN`）
- `getpass` 手输

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print('✅ logged in via Colab secret HF_TOKEN')
except Exception:
    from getpass import getpass
    login(token=getpass('HF token: '))
    print('✅ logged in via manual input')

## 4. 上传数据

本地先跑：
```bash
bash scripts/pack_for_colab.sh stage2
```
产出 `stage2_colab.zip`（只含 jsonl，几 MB）。

In [ ]:
from google.colab import files
up = files.upload()   # 选 stage2_colab.zip
!unzip -q -o stage2_colab.zip -d /content/
!ls /content/data/stage2_train/
%cd /content

## 5. 加载 Qwen2.5-0.5B（4-bit）+ LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_LEN = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    'Qwen/Qwen2.5-0.5B-Instruct',
    max_seq_length=MAX_LEN,
    load_in_4bit=True,
)
FastLanguageModel.for_training(model)

model = FastLanguageModel.get_peft_model(
    model,
    r=8, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
model.print_trainable_parameters()

## 6. 数据格式化（alpaca → chat template）

In [ ]:
from datasets import load_dataset

def to_chat(ex):
    user = f"{ex['instruction']}\n\n{ex['input']}"
    msgs = [{'role':'user', 'content': user},
            {'role':'assistant', 'content': ex['output']}]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False,
                                                   add_generation_prompt=False)}

ds = load_dataset('json', data_files={'train':'/content/data/stage2_train/train.jsonl',
                                       'val':  '/content/data/stage2_train/val.jsonl'})
ds = ds.map(to_chat, remove_columns=ds['train'].column_names)
print(ds)
print('样例:', ds['train'][0]['text'][:300])

## 7. 训练

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds['train'],
    eval_dataset=ds['val'],
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=600,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=20,
        eval_strategy='steps',
        eval_steps=100,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=3407,
        output_dir='outputs',
        dataset_text_field='text',
        max_seq_length=MAX_LEN,
    ),
)
trainer.train()

## 8. 推理验证

In [ ]:
FastLanguageModel.for_inference(model)
from transformers import TextStreamer

table_md = (
    '| 年份 | 营收(亿) | 净利润(亿) |\n'
    '|---|---|---|\n'
    '| 2022 | 100 | 15 |\n'
    '| 2023 | 120 | 18 |\n'
    '| 2024 | 135 | 22 |'
)
q = '哪一年净利润同比增长最大？'
msgs = [{'role':'user', 'content': f'基于表格回答：\n{table_md}\n\n问题：{q}'}]
prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
_ = model.generate(**inputs, max_new_tokens=200, streamer=TextStreamer(tokenizer, skip_prompt=True))

## 9. 保存 + 下载 adapter

In [ ]:
model.save_pretrained('stage2_adapter')
tokenizer.save_pretrained('stage2_adapter')
!ls -lh stage2_adapter/
!zip -q -r stage2_adapter.zip stage2_adapter/
!ls -lh stage2_adapter.zip

In [ ]:
from google.colab import files
files.download('stage2_adapter.zip')

---

## 🔁 回到本地部署

1. 解压到 `models/`：
   ```bash
   unzip stage2_adapter.zip -d models/
   mv models/stage2_adapter models/stage2_adapter_hf
   ```

2. 用 transformers + peft 合并：
   ```python
   from peft import PeftModel
   from transformers import AutoModelForCausalLM, AutoTokenizer
   base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
   model = PeftModel.from_pretrained(base, 'models/stage2_adapter_hf')
   model = model.merge_and_unload()
   model.save_pretrained('models/stage2_fused')
   tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
   tok.save_pretrained('models/stage2_fused')
   ```

3. 重启 / 热加载 Docker：
   ```bash
   docker compose restart api
   # 或热加载：
   curl -X POST http://localhost:8000/admin/reload \
     -H 'X-Admin-Key: '$ADMIN_API_KEY \
     -d '{"stage":2,"force":true}'
   ```
